In [4]:
# -*- coding: utf-8 -*-
"""
Batch Text-to-image inference script from prompt file
"""
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
import json
import argparse
import time
import torch
from transformers import AutoConfig, AutoTokenizer
from PIL import Image
import sys
from tqdm import tqdm
# sys.path.append(os.path.dirname(os.path.dirname(__file__)))

from config import SPECIAL_TOKENS
from model import LLaDAForMultiModalGeneration
from utils.generation_utils import setup_seed
from utils.image_utils import decode_vq_to_image, calculate_vq_params, add_break_line, encode_img_with_paint
from generators.image_generation_generator import generate_image
from utils.prompt_utils import generate_text_to_image_prompt, create_prompt_templates

    
checkpoint_path = 'Alpha-VLLM/Lumina-DiMOO'

# Load model and tokenizer
print(f"Loading model from {checkpoint_path}...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, trust_remote_code=True)
model = LLaDAForMultiModalGeneration.from_pretrained(
    checkpoint_path, torch_dtype=torch.bfloat16, device_map="auto",
)
model.eval()


Loading model from Alpha-VLLM/Lumina-DiMOO...


`torch_dtype` is deprecated! Use `dtype` instead!


Initializing MMadaModelLM with config: LLaDAConfig {
  "activation_type": "silu",
  "alibi": false,
  "alibi_bias_max": 8.0,
  "architectures": [
    "LLaDAForMultiModalGeneration"
  ],
  "attention_dropout": 0.0,
  "attention_layer_norm": false,
  "attention_layer_norm_with_affine": true,
  "auto_map": {
    "AutoConfig": "configuration_llada.LLaDAConfig",
    "AutoModel": "modeling_llada.LLaDAModelLM",
    "AutoModelForCausalLM": "modeling_llada.LLaDAModelLM"
  },
  "bias_for_layer_norm": false,
  "block_group_size": 1,
  "block_type": "llama",
  "d_model": 4096,
  "dtype": "bfloat16",
  "embedding_dropout": 0.0,
  "embedding_size": 134548,
  "eos_token_id": 126081,
  "flash_attention": false,
  "include_bias": false,
  "include_qkv_bias": false,
  "init_cutoff_factor": null,
  "init_device": "meta",
  "init_fn": "mitchell",
  "init_std": 0.02,
  "input_emb_norm": false,
  "layer_norm_type": "rms",
  "layer_norm_with_affine": true,
  "mask_token_id": 126336,
  "max_sequence_length": 

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LLaDAForMultiModalGeneration(
  (model): LLaDAModel(
    (transformer): ModuleDict(
      (wte): Embedding(134548, 4096)
      (emb_drop): Dropout(p=0.0, inplace=False)
      (ln_f): RMSLayerNorm()
      (blocks): ModuleList(
        (0-31): 32 x LLaDALlamaBlock(
          (dropout): Dropout(p=0.0, inplace=False)
          (act): SiLU()
          (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
          (rotary_emb): RotaryEmbedding()
          (attn_norm): RMSLayerNorm()
          (ff_norm): RMSLayerNorm()
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
      

In [ ]:
for n,p in model.named_parameters():
    print(n)

model.transformer.wte.weight
model.transformer.ln_f.weight
model.transformer.blocks.0.attn_out.weight
model.transformer.blocks.0.ff_out.weight
model.transformer.blocks.0.attn_norm.weight
model.transformer.blocks.0.ff_norm.weight
model.transformer.blocks.0.q_proj.weight
model.transformer.blocks.0.k_proj.weight
model.transformer.blocks.0.v_proj.weight
model.transformer.blocks.0.ff_proj.weight
model.transformer.blocks.0.up_proj.weight
model.transformer.blocks.1.attn_out.weight
model.transformer.blocks.1.ff_out.weight
model.transformer.blocks.1.attn_norm.weight
model.transformer.blocks.1.ff_norm.weight
model.transformer.blocks.1.q_proj.weight
model.transformer.blocks.1.k_proj.weight
model.transformer.blocks.1.v_proj.weight
model.transformer.blocks.1.ff_proj.weight
model.transformer.blocks.1.up_proj.weight
model.transformer.blocks.2.attn_out.weight
model.transformer.blocks.2.ff_out.weight
model.transformer.blocks.2.attn_norm.weight
model.transformer.blocks.2.ff_norm.weight
model.transformer

: 